In [4]:
%pip install keras-tuner

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
import keras
from keras import layers
import keras_tuner as kt

I0000 00:00:1793011835.933538  105974 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793011835.933880  105974 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793011835.959863  105974 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793011836.763792  105974 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [6]:
path = Path("./textes_films_horreur_comedie.csv")

df = pd.read_csv(path)

print("dimension:", df.shape)
display(df.head())
display(df["cible"].value_counts().sort_index())
display("valeurs manquante: ", df.isna().sum())
display("valeurs null: ", df.isnull().sum())
display(f"dupliqué: {df.duplicated().sum()}")
display(df.dtypes)

dimension: (2400, 3)


,titre,resume,cible
0,Chiedo asilo,Roberto has a new job as the teacher at a nurs...,0
1,Friday the 13th: A New Beginning,Five years after killing the goalie hockey-mas...,1
2,Stranger Than Fiction,Everybody knows that your life is a story. But...,0
3,Paris,"Pierre, a professional dancer, suffers from a ...",0
4,Sleepless,A retired detective with insomnia is called up...,1


cible
0    1200
1    1200
Name: count, dtype: int64

'valeurs manquante: '

titre     0
resume    0
cible     0
dtype: int64

'valeurs null: '

titre     0
resume    0
cible     0
dtype: int64

'dupliqué: 54'

titre       str
resume      str
cible     int64
dtype: object

In [7]:
nb_initial = len(df)

df["titre"] = (
    df["titre"].astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["resume"] = (
    df["resume"].astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

nb_doublons = df.duplicated().sum()
df = df.drop_duplicates().copy()

incomplets = (
    df["resume"].isna()
    | df["resume"].eq("").fillna(False)
    | df["cible"].isna()
)

nb_incomplets = incomplets.sum()
df = df.loc[~incomplets].reset_index(drop=True)

print("Lignes initiales: ", nb_initial)
print("Doublons supprimé: ", nb_doublons)
print("Lignes sans résumé ou cible supprimé: ", nb_incomplets)
print(len(df))

Lignes initiales:  2400
Doublons supprimé:  54
Lignes sans résumé ou cible supprimé:  0
2346


In [8]:
apprentissage, reste = train_test_split(
    df,
    test_size=0.4,
    stratify=df["cible"],
    random_state=42
)

test, production = train_test_split(
    reste,
    test_size=0.5,
    stratify=reste["cible"],
    random_state=42
)

for nom, partie in [
    ("apprentissage", apprentissage),
    ("test", test),
    ("production", production)
]:
    print(f"\n{nom} : {len(partie)} films")
    print(partie["cible"].value_counts(normalize=True))


apprentissage : 1407 films
cible
0    0.509595
1    0.490405
Name: proportion, dtype: float64

test : 469 films
cible
0    0.509595
1    0.490405
Name: proportion, dtype: float64

production : 470 films
cible
0    0.508511
1    0.491489
Name: proportion, dtype: float64


In [9]:
tfidf = TfidfVectorizer()

X_apprentissage = tfidf.fit_transform(apprentissage["resume"])
X_test = tfidf.transform(test["resume"])
X_production = tfidf.transform(production["resume"])

y_apprentissage = apprentissage["cible"]
y_test = test["cible"]
y_production = production["cible"]

print("Apprentissage: ", X_apprentissage.shape)
print("Test: ", X_test.shape)
print("Production: ", X_production.shape)

Apprentissage:  (1407, 14881)
Test:  (469, 14881)
Production:  (470, 14881)


In [10]:
graines = [0, 1, 2, 3, 4]
resultats = []

for graine in graines:
    X_train, X_val, y_train, y_val = train_test_split(
        X_apprentissage,
        y_apprentissage,
        test_size=0.2,
        stratify=y_apprentissage,
        random_state=graine
    )

    modeles = {
        "Régression logistique": LogisticRegression(random_state=graine),
        "Bayes naïf": MultinomialNB()
    }

    for nom, modele in modeles.items():
        modele.fit(X_train, y_train)
        predictions = modele.predict(X_val)

        resultats.append({
            "modele": nom,
            "graine": graine,
            "F1": f1_score(y_val, predictions)
        })

scores = pd.DataFrame(resultats)
display(scores)

,modele,graine,F1
0,Régression logistique,0,0.808824
1,Bayes naïf,0,0.865052
2,Régression logistique,1,0.829630
3,Bayes naïf,1,0.875862
4,Régression logistique,2,0.784615
5,Bayes naïf,2,0.846154
6,Régression logistique,3,0.773234
7,Bayes naïf,3,0.821918
8,Régression logistique,4,0.800000
9,Bayes naïf,4,0.875862


In [11]:
for graine in graines:
    keras.backend.clear_session()
    keras.utils.set_random_seed(graine)

    X_train, X_val, y_train, y_val = train_test_split(
        X_apprentissage,
        y_apprentissage,
        test_size=0.2,
        stratify=y_apprentissage,
        random_state=graine
    )

    X_train_dense = X_train.toarray().astype("float32")
    X_val_dense = X_val.toarray().astype("float32")

    reseau = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    reseau.compile(
        optimizer="adam",
        loss="binary_crossentropy"
    )

    reseau.fit(
        X_train_dense,
        y_train.to_numpy(dtype="float32"),
        validation_data=(
            X_val_dense,
            y_val.to_numpy(dtype="float32")
        ),
        epochs=10,
        batch_size=32,
        verbose=0
    )

    probabilites = reseau.predict(X_val_dense, verbose=0).ravel()
    predictions = (probabilites >= 0.5).astype(int)

    resultats.append({
        "modele":"Réseau simple",
        "graine": graine,
        "F1": f1_score(y_val, predictions)
    })

scores = pd.DataFrame(resultats)
display(scores)

E0000 00:00:1793011840.245160  105974 cuda_executor.cc:1690] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cuda_status.cc:68

W0000 00:00:1793011840.269923  105974 dlopen_checker.cc:38] Could not load dynamic library 'libcudart.so.12'; dlerror: libcudart.so.12: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/cuda/lib64
W0000 00:00:1793011840.269937  105974 dlopen_checker.cc:38] Could not load dynamic library 'libcublas.so.12'; dlerror: libcublas.so.12: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/cuda/lib64
W0000 00:00:1793011840.269939  105974 dlopen_checker.cc:38] Could not load dynamic library 'libcublasLt.so.12'; dlerror: libcublasLt.so.12: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/

,modele,graine,F1
0,Régression logistique,0,0.808824
1,Bayes naïf,0,0.865052
2,Régression logistique,1,0.829630
3,Bayes naïf,1,0.875862
4,Régression logistique,2,0.784615
5,Bayes naïf,2,0.846154
6,Régression logistique,3,0.773234
7,Bayes naïf,3,0.821918
8,Régression logistique,4,0.800000
9,Bayes naïf,4,0.875862


In [12]:
n_validation = len(y_val)

scores["marge_bruit"] = 1.96 * np.sqrt(
    scores["F1"] * (1 - scores["F1"]) / n_validation
)

comparaison = scores.groupby("modele").agg(
    F1_moyen=("F1", "mean"),
    ecart_type=("F1", "std"),
    marge_bruit_moyenne=("marge_bruit", "mean")
)

display(comparaison.round(4))

,F1_moyen,ecart_type,marge_bruit_moyenne
modele,,,
Bayes naïf,0.8570,0.0230,0.0407
Régression logistique,0.7993,0.0218,0.0467
Réseau simple,0.8525,0.0324,0.0411


In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    X_apprentissage,
    y_apprentissage,
    test_size=0.2,
    stratify=y_apprentissage,
    random_state=42
)

X_train_dense = X_train.toarray().astype("float32")
X_val_dense = X_val.toarray().astype("float32")

y_train_array = y_train.to_numpy(dtype="float32")
y_val_array = y_val.to_numpy(dtype="float32")

In [14]:
def construire_reseau(
        nb_couches,
        neurones,
        dropout,
        regularisation,
        taux_apprentissage
):
    modele = keras.Sequential([
        layers.Input(shape=(X_apprentissage.shape[1],))
    ])
    
    for _ in range(nb_couches):
        modele.add(layers.Dense(
            neurones,
            activation="relu",
            kernel_regularizer=keras.regularizers.l2(regularisation)
        ))
        modele.add(layers.Dropout(dropout))

    modele.add(layers.Dense(1, activation="sigmoid"))

    modele.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=taux_apprentissage
        ),
        loss="binary_crossentropy"
    )

    return modele

In [15]:
class RechercheF1(kt.RandomSearch):
    def run_trial(self, trial, X_train, y_train, X_val, y_val):
        keras.backend.clear_session()
        keras.utils.set_random_seed(42)

        hp = trial.hyperparameters

        modele = construire_reseau(
            nb_couches=hp.Choice("nb_couches", [1,2,3]),
            neurones=hp.Choice("neurones", [32, 64, 128]),
            dropout=hp.Choice("dropout", [0.0, 0.2, 0.4]),
            regularisation=hp.Choice(
                "regularisation", [0.0, 0.0001, 0.001]
            ),
            taux_apprentissage=hp.Choice(
                "taux_apprentissage", [0.0001, 0.001, 0.01]
            )
        )

        modele.fit(
            X_train,
            y_train,
            epochs=10,
            batch_size=32,
            verbose=0
        )

        probabilites = modele.predict(X_val, verbose=0).ravel()
        predictions = (probabilites >= 0.5).astype(int)

        return {"val_f1": float(f1_score(y_val, predictions))}

In [16]:
tuner = RechercheF1(
    objective=kt.Objective("val_f1", direction="max"),
    max_trials=20,
    seed=42,
    directory="recherches",
    project_name="keras_tuner_films",
    overwrite=True
)

tuner.search(
    X_train_dense,
    y_train_array,
    X_val_dense,
    y_val_array
)

Trial 20 Complete [00h 00m 04s]
val_f1: 0.8842105263157894

Best val_f1 So Far: 0.8857142857142857
Total elapsed time: 00h 01m 42s


In [17]:
meilleur_essai = tuner.oracle.get_best_trials(num_trials=1)[0]

meilleurs_reglages_keras = meilleur_essai.hyperparameters.values
meilleurs_f1_keras = meilleur_essai.score

print("Meilleurs réglages: ")
for nom, valeur in meilleurs_reglages_keras.items():
    print(f" {nom} : {valeur}")

print(f"\nMeilleur F1 de validation: {meilleurs_f1_keras:.4f}")

Meilleurs réglages: 
 nb_couches : 3
 neurones : 64
 dropout : 0.2
 regularisation : 0.0001
 taux_apprentissage : 0.0001

Meilleur F1 de validation: 0.8857


In [18]:
%pip install optuna ipywidgets

  Using cached ipywidgets-8.1.9-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.16-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.9-py3-none-any.whl (140 kB)
Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl (217 kB)
Using cached widgetsnbextension-4.0.16-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]
Note: you may need to restart the kernel to use updated packages.


In [19]:
import optuna

def objectif(trial):
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)

    nb_couches = trial.suggest_categorical("nb_couches", [1, 2, 3])
    neurones = trial.suggest_categorical("neurones", [32, 64, 128])
    dropout = trial.suggest_categorical("dropout", [0.0, 0.2, 0.4])
    regularisation = trial.suggest_categorical(
        "regularisation", [0.0, 0.0001, 0.001]
    )
    taux_apprentissage = trial.suggest_categorical(
        "taux_apprentissage", [0.0001, 0.001, 0.01]
    )

    modele = construire_reseau(
        nb_couches=nb_couches,
        neurones=neurones,
        dropout=dropout,
        regularisation=regularisation,
        taux_apprentissage=taux_apprentissage
    )

    modele.fit(
        X_train_dense,
        y_train_array,
        epochs=10,
        batch_size=32,
        verbose=0
    )

    probabilites = modele.predict(X_val_dense, verbose=0).ravel()
    predictions = (probabilites >= 0.5).astype(int)

    return float(f1_score(y_val_array, predictions))

etude = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

etude.optimize(objectif, n_trials=20)

meilleurs_reglages_optuna = etude.best_params
meilleurs_f1_optuna = etude.best_value

print("\nMeilleurs réponses: ")
for nom, valeur in meilleurs_reglages_optuna.items():
    print(f" {nom} : {valeur}")

print(f"\nMeilleur F1 de validation: {meilleurs_f1_optuna:.4f}")



[I 2026-10-26 11:52:40,740] A new study created in memory with name: no-name-7b27aacb-9421-4635-a040-146d4ba1e30b
[I 2026-10-26 11:52:43,069] Trial 0 finished with value: 0.7800829875518672 and parameters: {'nb_couches': 2, 'neurones': 32, 'dropout': 0.2, 'regularisation': 0.001, 'taux_apprentissage': 0.0001}. Best is trial 0 with value: 0.7800829875518672.
[I 2026-10-26 11:52:51,602] Trial 1 finished with value: 0.8452830188679246 and parameters: {'nb_couches': 3, 'neurones': 128, 'dropout': 0.4, 'regularisation': 0.0001, 'taux_apprentissage': 0.001}. Best is trial 1 with value: 0.8452830188679246.
[I 2026-10-26 11:52:56,197] Trial 2 finished with value: 0.8773234200743495 and parameters: {'nb_couches': 1, 'neurones': 64, 'dropout': 0.4, 'regularisation': 0.001, 'taux_apprentissage': 0.001}. Best is trial 2 with value: 0.8773234200743495.
[I 2026-10-26 11:53:05,345] Trial 3 finished with value: 0.8507462686567164 and parameters: {'nb_couches': 1, 'neurones': 128, 'dropout': 0.2, 'regu


Meilleurs réponses: 
 nb_couches : 3
 neurones : 64
 dropout : 0.0
 regularisation : 0.0
 taux_apprentissage : 0.0001

Meilleur F1 de validation: 0.8905


In [20]:
comparaison_recherches = pd.DataFrame({
    "outil": ["KeraTuner", "Optuna"],
    "F1_validation": [meilleurs_f1_keras, meilleurs_f1_optuna]
})

n = len(y_val_array)
p = comparaison_recherches["F1_validation"]

comparaison_recherches["marge_bruit"] = (
    1.96 * np.sqrt(p * (1 - p) / n)
)

display(comparaison_recherches.round(4))

,outil,F1_validation,marge_bruit
0,KeraTuner,0.8857,0.0371
1,Optuna,0.8905,0.0365


In [21]:
resultats_optimises = []

for graine in graines:
    keras.backend.clear_session()
    keras.utils.set_random_seed(graine)

    X_train, X_val, y_train, y_val = train_test_split(
        X_apprentissage,
        y_apprentissage,
        test_size=0.2,
        stratify=y_apprentissage,
        random_state=graine
    )

    X_train_dense = X_train.toarray().astype("float32")
    X_val_dense = X_val.toarray().astype("float32")

    modele = construire_reseau(**meilleurs_reglages_keras)

    modele.fit(
        X_train_dense,
        y_train.to_numpy(dtype="float32"),
        epochs=10,
        batch_size=32,
        verbose=0
    )

    probabilites = modele.predict(X_val_dense, verbose=0).ravel()
    predictions = (probabilites >= 0.5).astype(int)

    resultats_optimises.append({
        "modele": "Réseau optimisé",
        "graine": graine,
        "F1": f1_score(y_val, predictions)
    })

scores_optimises = pd.DataFrame(resultats_optimises)

scores_complets = pd.concat(
    [
        scores[["modele", "graine", "F1"]],
        scores_optimises
    ],
        ignore_index=True
)

scores_complets["marge_bruit"] = 1.96 * np.sqrt(
    scores_complets["F1"] * (1 - scores_complets["F1"]) / len(y_val)
)

comparaison_finale = scores_complets.groupby("modele").agg(
    F1_moyen=("F1", "mean"),
    ecart_type=("F1", "std"),
    marge_bruit_moyenne=("marge_bruit", "mean")
)

display(scores_optimises)
display(comparaison_finale.round(4))

,modele,graine,F1
0,Réseau optimisé,0,0.835249
1,Réseau optimisé,1,0.866426
2,Réseau optimisé,2,0.826568
3,Réseau optimisé,3,0.776923
4,Réseau optimisé,4,0.841699


,F1_moyen,ecart_type,marge_bruit_moyenne
modele,,,
Bayes naïf,0.8570,0.0230,0.0407
Régression logistique,0.7993,0.0218,0.0467
Réseau optimisé,0.8294,0.0329,0.0437
Réseau simple,0.8525,0.0324,0.0411


In [22]:
X_apprentissage_dense = X_apprentissage.toarray().astype("float32")
X_test_dense = X_test.toarray().astype("float32")
y_apprentissage_array = y_apprentissage.to_numpy(dtype="float32")

resultats_test = []

bayes_final = MultinomialNB()
bayes_final.fit(X_apprentissage, y_apprentissage)

predictions = bayes_final.predict(X_test)

resultats_test.append({
    "modele": "Bayes naïf",
    "graine": None,
    "F1": f1_score(y_test, predictions)
})

for graine in graines:
    for nom in ["Réseau simple", "Réseau optimisé"]:
        keras.backend.clear_session()
        keras.utils.set_random_seed(graine)

        if nom == "Réseau simple":
            reseau = construire_reseau(
                nb_couches=1,
                neurones=32,
                dropout=0.2,
                regularisation=0.0,
                taux_apprentissage=0.001
            )
        else:
            reseau = construire_reseau(**meilleurs_reglages_keras)

        reseau.fit(
            X_apprentissage_dense,
            y_apprentissage_array,
            epochs=10,
            batch_size=32,
            verbose=0
        )

        probabilites = reseau.predict(X_test_dense, verbose=0).ravel()
        predictions = (probabilites >= 0.5).astype(int)
        score = f1_score(y_test, predictions)

        resultats_test.append({
            "modele": nom,
            "graine": graine,
            "F1": score
        })

        if nom == "Réseau optimisé" and graine == 0:
            modele_final = reseau
            f1_test_final = float(score)

scores_test = pd.DataFrame(resultats_test)

scores_test["marge_bruit"] = 1.96 * np.sqrt(
    scores_test["F1"] * (1 - scores_test["F1"]) / len(y_test)
)

bilan_test = scores_test.groupby("modele").agg(
    F1_moyen=("F1", "mean"),
    ecart_type=("F1", "std"),
    marge_bruit_moyenne=("marge_bruit", "mean")
)

display(scores_test)
display(bilan_test.round(4))

print(f"F1 du modèle à sauvegarder: {f1_test_final:.4f}")

,modele,graine,F1,marge_bruit
0,Bayes naïf,NaN,0.875264,0.029904
1,Réseau simple,0.0,0.854664,0.031897
2,Réseau optimisé,0.0,0.857143,0.031670
3,Réseau simple,1.0,0.852814,0.032065
4,Réseau optimisé,1.0,0.868251,0.030610
5,Réseau simple,2.0,0.852814,0.032065
6,Réseau optimisé,2.0,0.868996,0.030537
7,Réseau simple,3.0,0.855263,0.031843
8,Réseau optimisé,3.0,0.859060,0.031492
9,Réseau simple,4.0,0.860310,0.031375


,F1_moyen,ecart_type,marge_bruit_moyenne
modele,,,
Bayes naïf,0.8753,NaN,0.0299
Réseau optimisé,0.8632,0.0053,0.0311
Réseau simple,0.8552,0.0031,0.0318


F1 du modèle à sauvegarder: 0.8571


In [23]:
import joblib, json, sklearn
from importlib.metadata import version

In [24]:
dossier_modele = Path("modele")
dossier_modele.mkdir(parents=True, exist_ok=True)

modele_final.save(dossier_modele / "reseau.keras")

joblib.dump(tfidf, dossier_modele / "tfidf.joblib")

metriques = {
    "version_modele": "1.0.0",
    "f1_test": f1_test_final,
    "taille_test": len(y_test),
    "versions_bibliotheques": {
        "keras": keras.__version__,
        "tensorflow": version("tensorflow"),
        "scikit-learn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "joblib": joblib.__version__
    }
}

with open(dossier_modele /"metrics.json", "w", encoding="utf-8") as fichier:
    json.dump(metriques, fichier, ensure_ascii=False, indent=4)

reference = apprentissage[["titre", "resume", "cible"]].sample(
    n=10,
    random_state=42
)

reference.to_csv(
    dossier_modele / "reference.csv",
    index=False,
    encoding="utf-8"
)

print("Fichiers sauvegardés: ")
for chemin in sorted(dossier_modele.iterdir()):
    print(chemin)

print(f"\nF1 du réseau sauvgardé: {f1_test_final:.4f}")

Fichiers sauvegardés: 
modele/metrics.json
modele/reference.csv
modele/reseau.keras
modele/tfidf.joblib

F1 du réseau sauvgardé: 0.8571


In [2]:
# Recharger les fichiers sauvegardés
reseau_charge = keras.models.load_model(dossier_modele / "reseau.keras")
tfidf_charge = joblib.load(dossier_modele / "tfidf.joblib")

with open(dossier_modele / "metrics.json", encoding="utf-8") as fichier:
    metriques_chargees = json.load(fichier)

# Afficher les couches du réseau et le nombre de paramètres
reseau_charge.summary()

# Afficher les informations enregistrées
print("\nMétadonnées du modèle :")
print(json.dumps(metriques_chargees, ensure_ascii=False, indent=4))

# Faire des prédictions sur l'échantillon de référence
reference = pd.read_csv(dossier_modele / "reference.csv")

X_reference = tfidf_charge.transform(reference["resume"])
probabilites = reseau_charge.predict(
    X_reference.toarray().astype("float32"),
    verbose=0
).ravel()

reference["probabilite_classe_1"] = probabilites
reference["prediction"] = (probabilites >= 0.5).astype(int)

display(reference[[
    "titre",
    "cible",
    "prediction",
    "probabilite_classe_1"
]])

NameError: name 'keras' is not defined